# 🧱 Prompt Templates in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **PromptTemplate basics** - build a reusable prompt with `{variables}`, and what `input_variables` really checks
2. **format() vs format_prompt() vs invoke()** - when you get a string and when you get a `PromptValue`
3. **Prompts in chains** - send templates to the model directly, in an LCEL chain, and with streaming
4. **ChatPromptTemplate** - prompts made of role-tagged messages; `from_template` vs `from_messages`
5. **PromptTemplate vs ChatPromptTemplate** - which one to use with which model
6. **Many inputs at once** - fill one template for several inputs and batch the calls

## Where this fits
`02_LLM_vs_ChatModel.ipynb` showed that models accept a string or a list of messages. This notebook
builds both from reusable templates. `04_Few_Shot_Composition_and_Serialization.ipynb` then adds
examples to templates, composes them from parts and saves them to files.

## Prerequisites
- `01_LangChain_Inputs_and_Outputs.ipynb` and `02_LLM_vs_ChatModel.ipynb` (reading)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

## 🔑 Quick Reference

| Method | Returns | Use Case |
|--------|---------|----------|
| `.format()` | `str` | Simple string formatting |
| `.format_prompt()` | `PromptValue` | When you need the intermediate object |
| `.invoke()` | `PromptValue` | For LCEL chains |

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the chat model every template is sent to
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")  # temperature defaults to 0

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📝 Part 1: PromptTemplate basics

Instead of hardcoding prompts like:
```python
SystemMessage(content="You are a helpful assistant that translates English to Spanish.")
```

You can make them dynamic:
```python
template = "You are a helpful assistant that translates {input_language} to {output_language}"
```

`PromptTemplate` holds such a string and fills the `{placeholders}` with Python's `str.format` syntax.

In [ ]:
# ============================================================================
# CREATING A PROMPTTEMPLATE
# ============================================================================
# Two ways to create:
# 1. Using the constructor with input_variables (declares them; see 1.1)
# 2. Using .from_template() (auto-detects variables)
# ============================================================================

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = """You are an expert in deep learning and PyTorch.
You answer queries by being brief, bright, and concise.

Query: {query}
"""

# Method 1: Explicit input_variables (readable, but not a check - see 1.1)
prompt_template = PromptTemplate(input_variables=['query'], template=template)
print("📋 Method 1 - Explicit variables:")
prompt_template.pretty_print()

# Method 2: Auto-detect from template (quick prototyping)
prompt_template = PromptTemplate.from_template(template)
print("\n📋 Method 2 - Auto-detected variables:")
prompt_template.pretty_print()

### 1.1 What `input_variables` does (and does not) do

Passing `input_variables=[...]` does **not** validate the template on its own: LangChain recomputes
the list from the template's placeholders and silently replaces yours. Turn on
`validate_template=True` to have a mismatch raise an error. A variable that is missing when you
*format* the template fails either way. The `try/except` blocks below catch errors on purpose: the
errors are the lesson.

In [ ]:
# ============================================================================
# INPUT VARIABLES: declared vs detected
# ============================================================================
TEMPLATE = "You are a helpful assistant that translates {input_language} to {output_language}."

explicit = PromptTemplate(template=TEMPLATE, input_variables=["input_language", "output_language"])
print(f"📋 Declared and detected : {explicit.input_variables}")
print(f"📝 Formatted             : {explicit.format(input_language='english', output_language='german')}")

# A typo in input_variables is silently replaced by what the template contains
typo = PromptTemplate(template=TEMPLATE, input_variables=["input_language", "output_langauge"])
print(f"📋 Typo is ignored       : {typo.input_variables}")

# validate_template=True compares the declared list with the template instead
try:
    PromptTemplate(
        template=TEMPLATE,
        input_variables=["input_language", "output_langauge"],
        validate_template=True,
    )
except ValueError as err:
    print(f"\n🚨 validate_template=True -> {type(err).__name__}:\n{err}")

# Formatting with a missing variable always fails
try:
    explicit.format(input_language="english")
except KeyError as err:
    print(f"\n🚨 Missing variable at format time -> KeyError: {err}")

---
## 🔄 Part 2: format() vs format_prompt() vs invoke()

`.format()` returns a plain string. `.format_prompt()` and `.invoke()` return a **PromptValue**: the
filled-in prompt as an object that can become either a string or a list of messages, depending on
which kind of model receives it.

In [ ]:
# ============================================================================
# .format() - Returns a String
# ============================================================================
# Use when you need the final prompt as a plain string
# ============================================================================

formatted_prompt = prompt_template.format(query="What is the best way to learn PyTorch?")

print("📝 Formatted prompt:")
print(formatted_prompt)
print(f"\n📊 Type: {type(formatted_prompt)}")  # <class 'str'>

In [ ]:
# ============================================================================
# .format_prompt() - Returns a PromptValue
# ============================================================================
# Use when you need the intermediate PromptValue object
# PromptValue can be converted to string or messages as needed
# ============================================================================

formatted_prompt_value = prompt_template.format_prompt(query="What is the best way to learn PyTorch?")

print("📝 PromptValue object:")
print(formatted_prompt_value)
print(f"\n📊 Type: {type(formatted_prompt_value)}")  # StringPromptValue

In [ ]:
# ============================================================================
# .invoke() - the Runnable entry point, also returns a PromptValue
# ============================================================================
# This is what an LCEL chain calls on the template. It takes a dict.
# ============================================================================
prompt_value = prompt_template.invoke({"query": "What is the best way to learn PyTorch?"})

print(f"📊 Type: {type(prompt_value).__name__}")
print(f"📝 As a string  : {prompt_value.to_string()!r}")
print(f"📝 As messages  : {prompt_value.to_messages()}")

### 2.1 Template vs value: blueprint vs instance

In [ ]:
# ============================================================================
# PROMPTTEMPLATE VS PROMPTVALUE
# ============================================================================
# PromptTemplate: A blueprint/factory (like a class)
# PromptValue: A specific instance with values filled in (like an object)
# ============================================================================

# Step 1: Create a PromptTemplate (the blueprint)
joke_template = PromptTemplate.from_template("Write a joke about {topic}")

# Step 2: Use format_prompt() to create a PromptValue (specific instance)
joke_value = joke_template.format_prompt(topic="penguins")

print("📋 PromptTemplate (blueprint):")
print(f"   Type: {type(joke_template)}")
print(f"   Variables: {joke_template.input_variables}")

print("\n📦 PromptValue (instance):")
print(f"   Type: {type(joke_value)}")
print(f"   Content: {joke_value.to_string()}")

#### 🧠 When to use each

- **Use PromptTemplate**:
    - When designing **reusable and parameterized prompts**.
    - For building **dynamic chains** with variables.
- **Use PromptValue**:
    - When you want to **pass a fully formatted prompt** to an LLM manually.
    - When chaining LLMs directly: llm.invoke(prompt_value)

---
## 🔗 Part 3: Prompts in chains

### 3.1 Sending prompts to the model
Both string prompts and PromptValue objects can be passed directly to `.invoke()`; the reply is an
`AIMessage` either way.

In [ ]:
# ============================================================================
# INVOKING THE MODEL WITH A STRING AND WITH A PROMPTVALUE
# ============================================================================
response = llm.invoke(formatted_prompt)  # str from .format()
print("🤖 Response (from string prompt):")
print(response.content)

response = llm.invoke(formatted_prompt_value)  # PromptValue from .format_prompt()
print("\n🤖 Response (from PromptValue):")
print(response.content)

print(f"\n📊 Response type: {type(response).__name__}")

### 3.2 Using prompts in LCEL chains

The **LangChain Expression Language (LCEL)** uses the `|` operator to chain components. The chain
calls `prompt_template.invoke(...)` for you, so you pass the variables as a dict.

In [ ]:
# ============================================================================
# LCEL CHAIN: prompt → LLM → output parser
# ============================================================================
# The pipe operator (|) connects components
# Each component's output becomes the next component's input
# ============================================================================

chain = prompt_template | llm | StrOutputParser()

# Now we pass a dict with the template variables
response = chain.invoke({"query": "What is the best way to learn PyTorch?"})

print("🔗 LCEL Chain Response:")
print(response)

### 3.3 Multi-input prompts

Templates can have **multiple variables** for more dynamic prompts.

In [ ]:
# ============================================================================
# MULTI-INPUT PROMPT EXAMPLE
# ============================================================================
# This function demonstrates using a prompt with multiple variables
# ============================================================================

def get_movie_information(movie_title: str, main_actor: str) -> str:
    """
    Generate a fictitious movie synopsis using an LLM.
    
    Args:
        movie_title: The title of the movie
        main_actor: The main actor of the movie
    Returns:
        Generated synopsis and genre
    """
    prompt = PromptTemplate(
        template="""Your task is to create a fictitious movie synopsis and genre for the following movie and main actor:
        Movie: {movie_title}
        Actor: {main_actor}""",
        input_variables=['movie_title', 'main_actor']
    )
    
    formatted_prompt = prompt.format(movie_title=movie_title, main_actor=main_actor)
    response = llm.invoke(formatted_prompt)
    
    return response.content

# Test the function
movie_info = get_movie_information("The Dark Knight", "Christian Bale")
print("🎬 Generated Movie Info:")
print(movie_info)

### 3.4 The same function with LCEL and streaming

`.stream()` yields the reply chunk by chunk. The function prints each chunk as it arrives and joins
them into the full text, so the model is called once.

In [ ]:
# ============================================================================
# LCEL VERSION WITH STREAMING
# ============================================================================
# .stream() provides token-by-token output for better UX
# ============================================================================

def get_movie_information_streaming(movie_title: str, main_actor: str) -> str:
    """Generate movie info with streaming output; return the full text."""
    prompt = PromptTemplate.from_template(template="""
        Your task is to create a fictitious movie synopsis and genre for:
        Movie: {movie_title}
        Actor: {main_actor}
        """)

    llm_chain = prompt | llm | StrOutputParser()

    # Stream output token by token, keeping each chunk
    print("🎬 Streaming response:")
    chunks = []
    for chunk in llm_chain.stream({"movie_title": movie_title, "main_actor": main_actor}):
        print(chunk, end="", flush=True)
        chunks.append(chunk)
    print("\n")

    return "".join(chunks)


result = get_movie_information_streaming(movie_title="Amritsar:1984", main_actor="Gurdaas Mann")
print(f"📊 Collected {len(result)} characters from the stream")

---
## 💬 Part 4: ChatPromptTemplate

`ChatPromptTemplate` creates **multi-turn prompts** in a chat format using **roles** like system,
user, and ai. Use it with chat models such as OpenAI's GPT models, Anthropic Claude or Gemini.

In [ ]:
# ============================================================================
# CHATPROMPTTEMPLATE EXAMPLE
# ============================================================================
# Use tuples of (role, content) to define messages
# Roles: "system", "user" (or "human"), "ai" (or "assistant")
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates text to French."),
    ("user", "Translate this sentence: {sentence}")
])

# .format() returns a string representation
formatted_chat_prompt = chat_prompt.format(sentence="I love programming.")
print("📝 Formatted (string):")
print(formatted_chat_prompt)

In [ ]:
# ============================================================================
# CHATPROMPTTEMPLATE: format() vs invoke() vs format_prompt()
# ============================================================================
print(f"📊 .format()        -> {type(formatted_chat_prompt).__name__}")

chat_prompt_value = chat_prompt.invoke({"sentence": "I love programming."})
print(f"📊 .invoke()        -> {type(chat_prompt_value).__name__}")
print(f"📊 .format_prompt() -> {type(chat_prompt.format_prompt(sentence='I love programming.')).__name__}")

print("\n📋 Messages inside the ChatPromptValue:")
for msg in chat_prompt_value.messages:
    print(f"  {type(msg).__name__}: {msg.content}")

### 4.1 `from_template` vs `from_messages`

| Aspect | `from_template` | `from_messages` |
|--------|-----------------|-----------------|
| **Input** | Single string | List of message tuples |
| **Roles** | Defaults to `HumanMessage` | Explicit control over roles |
| **System prompt** | Not directly supported | ✅ Supported |
| **Multi-turn context** | ❌ | ✅ Can include AI responses |
| **Use case** | Simple, single prompts | Complex chat interactions |

**Role shortcuts for `from_messages`:**

| Tuple Role | Message Class |
|------------|---------------|
| `"system"` | `SystemMessage` |
| `"human"` / `"user"` | `HumanMessage` |
| `"ai"` / `"assistant"` | `AIMessage` |

Under the hood, `from_template("Explain {topic}")` is the same as
`from_messages([("human", "Explain {topic}")])`. The cell below checks that.

In [ ]:
# ============================================================================
# from_template IS A ONE-HUMAN-MESSAGE from_messages (no model call)
# ============================================================================
simple = ChatPromptTemplate.from_template("Explain {topic}")
explicit_roles = ChatPromptTemplate.from_messages([("human", "Explain {topic}")])


def shape(template) -> list:
    return [(type(m).__name__, m.content) for m in template.invoke({"topic": "AI"}).to_messages()]


print(f"📋 from_template : {shape(simple)}")
print(f"📋 from_messages : {shape(explicit_roles)}")
print(f"✅ Same messages : {shape(simple) == shape(explicit_roles)}")

### 4.2 Conversation history inside a template

A template can carry fixed earlier turns (a `human` message and an `ai` reply) before the user's
real input. The model reads them as conversation history.

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, yet slightly quirky and cheeky AI bot. Your name is {name}."),
    ("human", "Yo! Wassup nephew."),
    ("ai", "As an AI language model, I am incapable of being your nephew."),
    ("human", "{user_input}"),
])

print(f"📋 Type: {type(template).__name__}")
print(f"📋 Input variables: {template.input_variables}")
print(f"📋 Message templates: {len(template.messages)}")

In [ ]:
# format_messages() fills the template and returns the list of messages
messages = template.format_messages(
    name="Robotalker",
    user_input="Talk robo to me!"
)
for msg in messages:
    print(f"  {type(msg).__name__}: {msg.content}")

print("\n🤖 Direct call:")
print(llm.invoke(messages).content)

# The same template in an LCEL chain
chain = template | llm | StrOutputParser()
print("\n🔗 LCEL chain:")
print(chain.invoke({"name": "Robotalker", "user_input": "Talk robo to me!"}))

### 4.3 Message objects and message prompt templates

`from_messages` also accepts message objects. A fixed `SystemMessage` has no variables; a
`HumanMessagePromptTemplate` is a message with placeholders.

In [ ]:
# ============================================================================
# SYSTEMMESSAGE + HUMANMESSAGEPROMPTTEMPLATE
# ============================================================================
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

system_message = SystemMessage(content="You are an OG language model who has good heart (operating system) but a bad user interface (you're super freaking rude).")
human_message = HumanMessagePromptTemplate.from_template("{text}")
template = ChatPromptTemplate.from_messages([system_message, human_message])

print(f"📋 Input variables: {template.input_variables}")  # only the human template's {text}
for msg in template.messages:
    print(f"  {type(msg).__name__}")

chain = template | llm | StrOutputParser()
print("\n🤖 Response:")
print(chain.invoke({"text": "That Sam I Am, I do not like that Sam I Am..."}))

---
## ⚖️ Part 5: PromptTemplate vs ChatPromptTemplate

| Aspect | `PromptTemplate` | `ChatPromptTemplate` |
|--------|------------------|----------------------|
| **Output type** | `StringPromptValue` (string) | `ChatPromptValue` (list of messages) |
| **Designed for** | Completion models (text-in, text-out) | Chat models (messages-in, message-out) |
| **System prompt** | ❌ Not supported | ✅ Supported |
| **Multiple roles** | ❌ Single string only | ✅ system, human, ai |
| **With ChatOpenAI** | Auto-converts to HumanMessage | Native message format |

**When to use each:**

| Use Case | Recommendation |
|----------|----------------|
| Simple prompts with ChatOpenAI | Either works ✅ |
| Need system prompt | `ChatPromptTemplate` |
| Need multi-turn/few-shot | `ChatPromptTemplate` |
| Working with completion models (legacy) | `PromptTemplate` |
| Building complex chat applications | `ChatPromptTemplate` |

For chat models, prefer `ChatPromptTemplate`: it supports system prompts and multiple roles and makes
the message structure explicit.

In [ ]:
# ============================================================================
# SAME TEXT, BOTH TEMPLATE TYPES
# ============================================================================
text_template = PromptTemplate.from_template("Explain {topic} in one sentence.")
chat_template = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")

for name, tpl in [("PromptTemplate", text_template), ("ChatPromptTemplate", chat_template)]:
    value = tpl.invoke({"topic": "AI"})
    print(f"📋 {name:<18} -> {type(value).__name__}: {value.to_messages()}")

# Both work with a chat model: the string is wrapped in a HumanMessage
print()
for name, tpl in [("PromptTemplate", text_template), ("ChatPromptTemplate", chat_template)]:
    answer = (tpl | llm | StrOutputParser()).invoke({"topic": "AI"})
    print(f"🤖 {name}: {answer}")

---
## 📚 Part 6: Many inputs at once

Fill one template per input and send all prompts in one `.batch()` call. Inside a chain,
`chain.batch(...)` fills the template for each input dict for you.

In [ ]:
# ============================================================================
# ONE TEMPLATE, SEVERAL INPUTS, ONE BATCH CALL
# ============================================================================
explain_template = PromptTemplate.from_template("Explain to me briefly about {topic} in {language}.")

inputs = [("Generative AI", "english"),
          ("Artificial Intelligence", "hindi"),
          ("Deep Learning", "german")]

prompts = [explain_template.format(topic=topic, language=language) for topic, language in inputs]
for p in prompts:
    print(f"📝 {p}")

responses = llm.batch(prompts)
for (topic, language), response in zip(inputs, responses):
    print(f"\n🌍 {topic} ({language}):")
    print(response.content)

In [ ]:
# ============================================================================
# A CHAT TEMPLATE WITH HISTORY, BATCHED THROUGH A CHAIN
# ============================================================================
messages = [
        ("system", "Act as an expert in real estate and provide brief answers"),
        ("human", "what is your name?"),
        ("ai", "my name is AIBot"),
        ("human", "{user_prompt}"),
]
real_estate_template = ChatPromptTemplate.from_messages(messages)
chain = real_estate_template | llm | StrOutputParser()

text_prompts = ["what is your name?", "explain commercial real estate to me"]
answers = chain.batch([{"user_prompt": p} for p in text_prompts])

for question, answer in zip(text_prompts, answers):
    print(f"❓ {question}")
    print(f"🤖 {answer}")
    print("-----")

The first answer comes from the template's own history: the `ai` turn already said the name is
AIBot, so the model repeats it.

---
## 📝 Summary

In this notebook, we learned:

### 1. PromptTemplate basics
- `PromptTemplate` fills `{placeholders}` with `str.format`; build it with `.from_template()` or the constructor
- `input_variables` is recomputed from the template; only `validate_template=True` turns a mismatch into an error

### 2. format() vs format_prompt() vs invoke()
- `.format()` returns a `str`; `.format_prompt()` and `.invoke()` return a `PromptValue`
- A `PromptValue` becomes a string or a message list, whichever the model needs

### 3. Prompts in chains
- `llm.invoke()` accepts a string or a `PromptValue`; `prompt | llm | StrOutputParser()` takes the variables as a dict
- `.stream()` yields chunks; join them instead of calling the model a second time

### 4. ChatPromptTemplate
- Built from `(role, content)` tuples or message objects / message prompt templates
- `from_template(...)` is `from_messages([("human", ...)])`; use `from_messages` for system prompts and history

### 5. PromptTemplate vs ChatPromptTemplate
- `PromptTemplate` -> `StringPromptValue`; `ChatPromptTemplate` -> `ChatPromptValue`
- Both work with chat models; prefer `ChatPromptTemplate` for them

### 6. Many inputs at once
- Format one prompt per input and call `llm.batch(prompts)`, or call `chain.batch([...])` on a chain

### Next Steps
- **`04_Few_Shot_Composition_and_Serialization.ipynb`**: examples in templates, composing templates from parts, saving them to files